# Data Pipelines: Designing ETL at Scale

## Learning Objectives
- Understand batch vs streaming pipeline architecture
- Build production Airflow DAGs with error handling
- Design data pipelines for 1M+ events/day
- Know orchestration tools and when to use each

## Interview Questions This Covers
- "Design a data pipeline for [system] processing 1M events/day"
- "Your batch pipeline failed. How do you debug?"
- "Compare batch vs streaming pipelines"
- "Optimize a slow data pipeline"

## Basic Implementation: Simple Airflow DAG

Minimal example showing core concepts

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import pandas as pd

# Define tasks
def extract_data():
    """Extract data from source"""
    data = pd.DataFrame({'user_id': [1, 2, 3], 'amount': [10, 20, 30]})
    return data.to_json()

def transform_data(ti):
    """Transform and aggregate"""
    data_json = ti.xcom_pull(task_ids='extract')
    data = pd.read_json(data_json)
    data['amount_scaled'] = data['amount'] * 1.1
    return data.to_json()

# Define DAG
dag = DAG(
    'simple_pipeline',
    start_date=datetime(2026, 5, 16),
    schedule_interval='@daily',
    catchup=False
)

# Define tasks
extract = PythonOperator(task_id='extract', python_callable=extract_data, dag=dag)
transform = PythonOperator(task_id='transform', python_callable=transform_data, dag=dag)

# Set dependencies
extract >> transform

print("✓ Basic DAG created")

## Advanced Implementation: Production Airflow with Error Handling

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.utils.task_group import TaskGroup
from airflow.exceptions import AirflowException
import time
import logging
from datetime import datetime

logger = logging.getLogger(__name__)

def extract_with_retry(max_retries=3):
    """Extract with automatic retry and exponential backoff"""
    for attempt in range(max_retries):
        try:
            logger.info(f"Extracting data, attempt {attempt + 1}")
            # Simulate extraction
            data_rows = 1000
            logger.info(f"✓ Extracted {data_rows} rows")
            return {'rows': data_rows, 'timestamp': datetime.now().isoformat()}
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                logger.warning(f"Extraction failed: {e}. Retrying in {wait_time}s")
                time.sleep(wait_time)
            else:
                raise AirflowException(f"Extraction failed after {max_retries} attempts")

def validate_data(ti):
    """Validate extracted data meets quality requirements"""
    data_info = ti.xcom_pull(task_ids='ingestion.extract')
    if data_info['rows'] < 100:
        raise AirflowException(f"Data validation failed: {data_info['rows']} rows (minimum 100)")
    logger.info(f"✓ Data validation passed: {data_info['rows']} rows")

def transform_data(ti):
    """Transform with performance monitoring"""
    data_info = ti.xcom_pull(task_ids='ingestion.extract')
    rows = data_info['rows']
    
    start_time = time.time()
    # Simulate transformation
    transformed_rows = rows
    duration = time.time() - start_time
    
    throughput = transformed_rows / duration if duration > 0 else 0
    logger.info(f"✓ Transformed {transformed_rows} rows in {duration:.2f}s ({throughput:.0f} rows/s)")
    return {'rows_processed': transformed_rows, 'duration': duration}

# Production DAG with task groups
dag = DAG(
    'production_pipeline',
    start_date=datetime(2026, 1, 1),
    schedule_interval='@daily',
    default_view='graph',
    catchup=False,
    tags=['production', 'data-pipeline']
)

with TaskGroup(name='ingestion', dag=dag) as ingestion:
    extract = PythonOperator(
        task_id='extract',
        python_callable=extract_with_retry
    )

with TaskGroup(name='quality', dag=dag) as quality:
    validate = PythonOperator(
        task_id='validate',
        python_callable=validate_data
    )

transform = PythonOperator(
    task_id='transform',
    python_callable=transform_data,
    dag=dag
)

# Define workflow
ingestion >> quality >> transform

logger.info("✓ Production pipeline DAG created")

## Real-World Example 1: Netflix Feature Pipeline

Batch pipeline for recommendation features

In [ ]:
# Netflix: Ingest 1B+ events/day, compute features for recommendations
# Key: separate batch (expensive embeddings) from real-time (session context)

def netflix_batch_pipeline():
    """Daily batch job: user embeddings, content similarity"""
    # 1. Ingest yesterday's viewing events (100GB+)
    events_ingested = 1_000_000_000  # 1B events
    
    # 2. Aggregate: user × content × watch_time
    users = 100_000_000  # 100M users
    avg_features_per_user = 10  # features per user
    
    # 3. Compute expensive features (embeddings)
    user_embeddings = users * 1000  # 1000-dim vectors
    content_similarity = 10_000 * 10_000  # content × content similarity matrix
    
    print(f"✓ Netflix batch: {events_ingested:,} events → {users:,} users")
    print(f"  Features: user embeddings, content similarity")
    print(f"  Cost: $1000/day (compute for embeddings)")
    print(f"  Freshness: daily (acceptable for embeddings)")
    return {'batch_features': 'ready'}

def netflix_streaming_features():
    """Real-time: current user session context"""
    # Event: user watches show X
    # Compute immediately: what's user watching NOW?
    current_watching = 50_000  # concurrent streams
    latency_ms = 5  # compute in <5ms
    
    print(f"✓ Netflix streaming: {current_watching:,} concurrent users")
    print(f"  Features: current session context")
    print(f"  Latency: {latency_ms}ms (must be real-time)")
    print(f"  Freshness: milliseconds")
    return {'streaming_features': 'ready'}

netflix_batch_pipeline()
print()
netflix_streaming_features()

## Real-World Example 2: Uber Surge Pricing Pipeline

Real-time features for pricing decisions

In [ ]:
# Uber: Real-time surge pricing requires fresh demand/supply data
# Latency constraint: <100ms (pricing decision needed immediately)

import time

def uber_real_time_pricing():
    """Real-time feature computation for surge pricing"""
    
    # Features needed for pricing decision
    features = {
        'demand': 500,  # ride requests in past 5 min
        'supply': 200,  # available drivers
        'surge_ratio': 500 / 200,  # 2.5x surge
        'avg_wait_time': 8,  # minutes
    }
    
    # Compute time budgets
    compute_start = time.time()
    
    # 1. Fetch demand from Kafka stream (<20ms)
    demand_fetch_ms = 5
    time.sleep(demand_fetch_ms / 1000)
    
    # 2. Fetch supply from cache (<10ms)
    supply_fetch_ms = 8
    time.sleep(supply_fetch_ms / 1000)
    
    # 3. Compute surge ratio (<30ms)
    ratio_compute_ms = 2
    time.sleep(ratio_compute_ms / 1000)
    
    total_ms = (time.time() - compute_start) * 1000
    
    print(f"✓ Uber surge pricing pipeline:")
    print(f"  Demand: {features['demand']} requests")
    print(f"  Supply: {features['supply']} drivers")
    print(f"  Surge ratio: {features['surge_ratio']:.1f}x")
    print(f"  Total latency: {total_ms:.0f}ms (budget: <100ms)")
    print(f"  ✓ Within SLA")
    
    return features

pricing = uber_real_time_pricing()

## Real-World Example 3: Stripe Fraud Detection Pipeline

Hybrid batch + streaming for fraud detection

In [ ]:
# Stripe: Fraud detection needs both historical (batch) and real-time (streaming) features
# Batch: historical frauds → train model
# Streaming: real-time transaction velocity → detect anomalies

def stripe_fraud_pipeline():
    """Hybrid pipeline for fraud detection"""
    
    print("Stripe Fraud Detection Pipeline:")
    print()
    
    # Batch component
    print("1. Batch Pipeline (Daily Training):")
    batch_stats = {
        'transactions_processed': 1_000_000,
        'confirmed_frauds': 5_000,
        'fraud_rate': 0.5,  # 0.5%
        'features_computed': ['user_history', 'merchant_risk', 'amount_anomaly'],
        'execution_time_hours': 2,
        'cost': '$500/day',
    }
    for key, value in batch_stats.items():
        print(f"   {key}: {value}")
    
    print()
    print("2. Streaming Component (Real-time Serving):")
    streaming_stats = {
        'transactions_per_second': 10_000,
        'real_time_features': ['transaction_velocity', 'merchant_anomaly', 'geographic_anomaly'],
        'scoring_latency_ms': 50,
        'storage': 'Redis (hot cache)',
    }
    for key, value in streaming_stats.items():
        print(f"   {key}: {value}")
    
    print()
    print("3. Key Decision: Batch vs Streaming")
    print("   Batch (expensive, historical): User embeddings, merchant risk scores")
    print("   Streaming (cheap, real-time): Current transaction velocity")
    print("   → Combine at inference time for fraud score")
    
stripe_fraud_pipeline()

## Interview Scenario: Design Fraud Detection Pipeline

**Question:** "You're at a payment company processing 1M transactions/day. Design a data pipeline for fraud detection that needs to flag fraudulent transactions in <100ms. Fraud labels are confirmed 5 days after the transaction. How would you architect this?"

**Solution Walkthrough:**

In [ ]:
# Interview Solution Structure

print("INTERVIEW ANSWER STRUCTURE:")
print()
print("1. CLARIFYING CONSTRAINTS")
print("   - 1M transactions/day = ~10 transactions/second")
print("   - Must score in <100ms (real-time serving)")
print("   - Fraud labels delayed 5 days (training data lag)")
print("   - Implies: batch training + streaming inference")
print()

print("2. ARCHITECTURE (Batch + Streaming)")
print("   Batch (Daily):")
print("   - Ingest confirmed fraud labels from past 5 days")
print("   - Compute features: user history, merchant risk, amount patterns")
print("   - Train fraud detection model")
print("   - Deploy new model version")
print()
print("   Streaming (Real-time):")
print("   - Real-time transaction events → Kafka")
print("   - Compute cheap features: transaction velocity, geographic anomaly")
print("   - Cache features in Redis (<5ms lookup)")
print("   - Score with model + real-time features in <100ms")
print()

print("3. KEY TRADE-OFFS")
print("   - Batch model: accurate (trained on 5 days of data) but stale")
print("   - Streaming features: fresh (real-time velocity) but simple")
print("   - Ensemble: combine model + rules for coverage")
print()

print("4. MONITORING & ITERATION")
print("   - Monitor: fraud detection rate, false positive rate")
print("   - Alert if accuracy drops below 95%")
print("   - Trigger retraining if fraud patterns shift")
print()

print("WHY THIS ANSWER WINS:")
print("✓ Separates batch (training) from streaming (serving)")
print("✓ Handles label delay constraint (5-day lag)")
print("✓ Achieves <100ms latency requirement")
print("✓ Discusses monitoring and iteration")
print("✓ Shows understanding of cost/latency trade-offs")

## Key Takeaways for Interviews

**What Interviewers Listen For:**
1. Do you understand batch vs streaming trade-offs?
2. Can you design for the constraints (throughput, latency, cost)?
3. Do you know common tools (Airflow, Kubeflow, Spark)?
4. Can you discuss failure recovery and monitoring?
5. Do you separate concerns (orchestration vs transformation)?

**Common Follow-ups:**
- "Your pipeline failed. How do you debug?"
  → Answer: Check orchestrator logs → task logs → data validation → resource issues
- "Pipeline is too slow. Optimize it."
  → Answer: Profile each stage → find bottleneck → optimize that stage first
- "Cost is too high. How do you cut it?"
  → Answer: Filter data early → compress → parallelize better → schedule off-peak
- "How do you prevent training-serving skew?"
  → Answer: Version features → validate data contracts → monitor freshness

**2-Minute Elevator Pitch:**
"Data pipelines move data from sources to models. Batch for training (historical data, slow), streaming for serving (real-time features, fast). Most systems need both. Separate orchestration (Airflow) from transformation (Spark). Include validation, monitoring, and error handling from day one."

**10-Minute Deep Dive:**
See real-world examples above (Netflix, Uber, Stripe). Each has different latency/cost requirements driving architectural choices. The key is understanding trade-offs and making explicit decisions.

## Next Steps

- Read `concepts/01-data-pipelines.md` for detailed theory
- Study the case studies (Stripe, Netflix, Uber)
- Practice answering interview questions in `interview-questions/questions.json`
- Do mock interviews discussing your designs